<a href="https://colab.research.google.com/github/SRakshithaReddy/CDR-Forensic-Analysis/blob/main/forensic_dataset_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Process Summary: Developed a modular Python preprocessing pipeline for raw numerical forensic features (timing_jitter_ms, entropy_score, signal_amplitude, syscall_rate). First, the script audits missing values, skew, and extreme spikes so we understand data quality before transforming anything. Next, it splits the dataset into training and test subsets before fitting imputers or scalers, preventing data leakage from test records into training. Missing values are filled using median imputation (SimpleImputer(strategy="median")) to resist distortion from heavy tails or anomalies, and feature vectors are normalized using a configurable scaler (RobustScaler, StandardScaler, or MinMaxScaler). Code is documented with docstrings, type hints, and clear comments; AI tools (ChatGPT / GitHub Copilot) were used to accelerate boilerplate and docstring drafting while manually verifying edge cases like missing columns and train/test isolation.

In [7]:
from __future__ import annotations

import logging
from typing import Literal

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

ScalerType = Literal["standard", "minmax", "robust"]
ImputeStrategy = Literal["median", "mean", "most_frequent"]

In [8]:
def generate_sample_forensic_dataset(
    n_samples: int = 250,
    random_state: int = 42,
) -> pd.DataFrame:
    """Generate a synthetic numerical forensic dataset with missing values and skew."""
    rng = np.random.default_rng(random_state)

    timing_jitter_ms = rng.exponential(scale=12.0, size=n_samples)
    entropy_score = np.clip(rng.normal(loc=5.4, scale=1.3, size=n_samples), 0.0, 8.0)
    signal_amplitude = rng.lognormal(mean=2.1, sigma=0.65, size=n_samples)
    syscall_rate = rng.normal(loc=140.0, scale=35.0, size=n_samples)

    # Inject anomalous spikes into ~10% of records
    anomaly_idx = rng.choice(n_samples, size=max(5, n_samples // 10), replace=False)
    timing_jitter_ms[anomaly_idx] *= 6.0
    signal_amplitude[anomaly_idx] *= 4.5

    is_anomalous = np.zeros(n_samples, dtype=int)
    is_anomalous[anomaly_idx] = 1

    df = pd.DataFrame(
        {
            "timing_jitter_ms": timing_jitter_ms,
            "entropy_score": entropy_score,
            "signal_amplitude": signal_amplitude,
            "syscall_rate": syscall_rate,
            "is_anomalous": is_anomalous,
        }
    )

    # Simulate ~8% missing values per numerical feature
    feature_cols = [
        "timing_jitter_ms",
        "entropy_score",
        "signal_amplitude",
        "syscall_rate",
    ]
    for col in feature_cols:
        missing_mask = rng.random(n_samples) < 0.08
        df.loc[missing_mask, col] = np.nan

    logger.info("Dataset created with shape %s", df.shape)
    return df


def audit_numerical_dataset(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Summarize missing counts, missing percentages, and spread per numerical feature."""
    rows = []
    total = len(df)

    for col in feature_cols:
        s = df[col]
        missing_cnt = int(s.isna().sum())
        missing_pct = (missing_cnt / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "feature": col,
                "missing_cnt": missing_cnt,
                "missing_pct": round(missing_pct, 2),
                "mean": round(float(s.mean(skipna=True)), 4),
                "median": round(float(s.median(skipna=True)), 4),
                "std": round(float(s.std(skipna=True)), 4),
                "min": round(float(s.min(skipna=True)), 4),
                "max": round(float(s.max(skipna=True)), 4),
            }
        )

    return pd.DataFrame(rows)

In [9]:
def fit_imputer(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    strategy: ImputeStrategy = "median",
) -> SimpleImputer:
    """Fit SimpleImputer strictly on the training numerical features."""
    imputer = SimpleImputer(strategy=strategy)
    imputer.fit(train_df[feature_cols])
    logger.info("Fitted SimpleImputer with strategy='%s'", strategy)
    return imputer


def apply_imputer(
    df: pd.DataFrame,
    imputer: SimpleImputer,
    feature_cols: list[str],
) -> pd.DataFrame:
    """Transform numerical columns using a pre-fitted imputer without mutating input."""
    df_out = df.copy()
    transformed = imputer.transform(df_out[feature_cols])
    df_out[feature_cols] = pd.DataFrame(
        transformed,
        columns=feature_cols,
        index=df_out.index,
    )
    return df_out

In [10]:
def _get_scaler(scaler_type: ScalerType):
    """Instantiate the requested scikit-learn scaler."""
    if scaler_type == "standard":
        return StandardScaler()
    if scaler_type == "minmax":
        return MinMaxScaler()
    if scaler_type == "robust":
        return RobustScaler()
    raise ValueError(f"Unsupported scaler_type: {scaler_type}")


def fit_scaler(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    scaler_type: ScalerType = "robust",
):
    """Fit a feature scaler strictly on imputed training data."""
    scaler = _get_scaler(scaler_type)
    scaler.fit(train_df[feature_cols])
    logger.info("Fitted '%s' scaler on %d features", scaler_type, len(feature_cols))
    return scaler


def apply_scaler(
    df: pd.DataFrame,
    scaler,
    feature_cols: list[str],
) -> pd.DataFrame:
    """Scale numerical columns using a pre-fitted scaler without mutating input."""
    df_out = df.copy()
    scaled = scaler.transform(df_out[feature_cols])
    df_out[feature_cols] = pd.DataFrame(
        scaled,
        columns=feature_cols,
        index=df_out.index,
    )
    return df_out

In [11]:
def preprocess_forensic_pipeline(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str | None = None,
    impute_strategy: ImputeStrategy = "median",
    scaler_type: ScalerType = "robust",
    test_size: float = 0.2,
    random_state: int = 42,
) -> dict[str, pd.DataFrame]:
    """Split data first, impute missing values, and scale feature vectors."""
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing expected columns: {missing_cols}")

    stratify_col = df[target_col] if (target_col and target_col in df.columns) else None

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_col,
    )

    imputer = fit_imputer(train_df, feature_cols=feature_cols, strategy=impute_strategy)
    train_imputed = apply_imputer(train_df, imputer=imputer, feature_cols=feature_cols)
    test_imputed = apply_imputer(test_df, imputer=imputer, feature_cols=feature_cols)

    scaler = fit_scaler(train_imputed, feature_cols=feature_cols, scaler_type=scaler_type)
    train_scaled = apply_scaler(train_imputed, scaler=scaler, feature_cols=feature_cols)
    test_scaled = apply_scaler(test_imputed, scaler=scaler, feature_cols=feature_cols)

    result: dict[str, pd.DataFrame] = {
        "X_train_processed": train_scaled[feature_cols],
        "X_test_processed": test_scaled[feature_cols],
    }
    if target_col and target_col in df.columns:
        result["y_train"] = train_df[[target_col]]
        result["y_test"] = test_df[[target_col]]

    logger.info(
        "Pipeline finished | Train: %d rows | Test: %d rows | Train NaNs left: %d",
        len(train_scaled),
        len(test_scaled),
        int(result["X_train_processed"].isna().sum().sum()),
    )
    return result

In [12]:
feature_cols = [
    "timing_jitter_ms",
    "entropy_score",
    "signal_amplitude",
    "syscall_rate",
]

raw_df = generate_sample_forensic_dataset(n_samples=250, random_state=42)

print("=== 1. Pre-Processing Audit ===")
display(audit_numerical_dataset(raw_df, feature_cols=feature_cols))

processed = preprocess_forensic_pipeline(
    df=raw_df,
    feature_cols=feature_cols,
    target_col="is_anomalous",
    impute_strategy="median",
    scaler_type="robust",
    test_size=0.2,
    random_state=42,
)

print("\n=== 2. Scaled Training Features (Top 5 Rows) ===")
display(processed["X_train_processed"].head().round(4))

print("\n=== 3. Scaled Test Features (Top 5 Rows) ===")
display(processed["X_test_processed"].head().round(4))

=== 1. Pre-Processing Audit ===


,feature,missing_cnt,missing_pct,mean,median,std,min,max
0,timing_jitter_ms,15,6.0,18.5626,9.8529,30.9267,0.0976,200.4946
1,entropy_score,20,8.0,5.4657,5.5600,1.2916,2.0633,8.0000
2,signal_amplitude,14,5.6,13.1806,8.5965,17.3949,1.1889,134.6017
3,syscall_rate,25,10.0,140.0727,141.5474,34.4495,12.3056,251.2599



=== 2. Scaled Training Features (Top 5 Rows) ===


,timing_jitter_ms,entropy_score,signal_amplitude,syscall_rate
40,0.5314,0.3015,0.2165,0.9638
19,-0.3166,0.0000,-0.0522,0.7901
222,0.2657,-0.6105,-0.6514,-1.5807
93,-0.3937,-0.8227,-0.6400,-0.4109
69,0.1841,1.5894,-0.0845,0.2414



=== 3. Scaled Test Features (Top 5 Rows) ===


,timing_jitter_ms,entropy_score,signal_amplitude,syscall_rate
60,0.3008,0.0000,2.5740,0.3838
169,1.2602,-0.8298,1.3515,0.5823
195,2.5044,-0.8059,-0.3696,1.0157
137,3.0253,-0.0446,11.4370,0.2106
53,0.5142,0.1433,0.0206,1.0781
